In [11]:
# =========================================================
# STEP 4A – TOPIC MODELING (UMAP + HDBSCAN + BERTopic)
# =========================================================
import numpy as np
import pandas as pd
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
import matplotlib.pyplot as plt

# -------- CONFIG --------
EMB_FILE  = "embeddings_e5.npy"              # embedding 1024 chiều
DATA_FILE = "comments_weaklabel.csv"      # văn bản + weak_label

# -------- LOAD DATA --------
X  = np.load(EMB_FILE)
df = pd.read_csv(DATA_FILE, encoding="utf-8-sig")
texts = df["text_clean"].fillna("").astype(str).tolist()

print(f"✅ Loaded {len(texts)} comments | Embedding shape: {X.shape}")

# =========================================================
# 1) UMAP – Giảm chiều embedding 1024 → 15
# =========================================================
umap_model = UMAP(
    n_components=15,        #
    n_neighbors=30,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# =========================================================
# 2) HDBSCAN – Gom cụm sau khi giảm chiều
# =========================================================
hdbscan_model = HDBSCAN(
    min_cluster_size=500,       # tăng để giảm noise
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

# =========================================================
# 3) BERTopic – topic modeling
# =========================================================
topic_model = BERTopic(
    language="multilingual",
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    nr_topics=None,     # để thuật toán tự động sinh số topic
    verbose=True
)

print("🔹 Running BERTopic...")
topics, probs = topic_model.fit_transform(texts, X)

# Gán topic ID vào dataframe
df["topic_id"] = topics

# Lấy bảng thống kê topic
topic_info = topic_model.get_topic_info()

# =========================================================
# Bổ sung mô tả cho từng topic
topic_descriptions = {}
for topic_id in df["topic_id"].unique():
    if topic_id == -1:
        topic_descriptions[topic_id] = "Noise/Không có mô tả"
        continue
    
    words = topic_model.get_topic(topic_id)
    
    # Lấy 5 từ đầu tiên làm mô tả cho topic
    if not words:
        topic_descriptions[topic_id] = "Không có mô tả"
    else:
        # Cải tiến mô tả topic theo từ khóa và nội dung đã được cải tiến
        description = ", ".join([word[0] for word in words[:5]])  # Lấy 5 từ đầu tiên
        topic_descriptions[topic_id] = description  # Tùy chỉnh mô tả nếu cần

# Gán mô tả cho từng topic vào dataframe
df["topic_description"] = df["topic_id"].map(topic_descriptions)

# Output ra kết quả theo định dạng yêu cầu
for topic_id, description in topic_descriptions.items():
    print(f"Topic {topic_id} → {description}")

# =========================================================
# SAVE RESULTS
# =========================================================
df.to_csv("comments_with_topics_and_descriptions.csv", index=False, encoding="utf-8-sig")
topic_info.to_csv("topic_summary.csv", index=False, encoding="utf-8-sig")

print("✅ Saved: comments_with_topics_and_descriptions.csv & topic_summary.csv")
print(topic_info.head())

# =========================================================
# VISUALIZATION
# =========================================================
try:
    fig = topic_model.visualize_topics()
    fig.write_html("bertopic_topics.html")
    print("📊 Saved: bertopic_topics.html (interactive)")
except Exception as e:
    print("⚠️ Visualization skipped:", e)

2025-12-07 04:19:18,721 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


✅ Loaded 186744 comments | Embedding shape: (186744, 1024)
🔹 Running BERTopic...


2025-12-07 04:26:09,857 - BERTopic - Dimensionality - Completed ✓
2025-12-07 04:26:09,871 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-07 04:27:21,692 - BERTopic - Cluster - Completed ✓
2025-12-07 04:27:21,714 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-07 04:27:23,298 - BERTopic - Representation - Completed ✓


Topic 5 → video, camera, chụp, review, ảnh
Topic 4 → neutral, tiền, không, có, mà
Topic -1 → Noise/Không có mô tả
Topic 16 → tiền, giá, nhiêu, bao, mua
Topic 3 → iphone, android, apple, ip, samsung
Topic 0 → pro, 15, 16, 14, 13
Topic 13 → samsung, sam, sony, sung, samfan
Topic 23 → neutral, iphone, ip, apple, samsung
Topic 12 → xiaomi, phần, không, nó, hãng
Topic 1 → s23, ultra, s24, s25, s23u
Topic 9 → x8, find, oppo, pro, x7
Topic 39 → negative, quá, ếch, giếng, mua
Topic 14 → 120hz, 60hz, màn, 120, 60
Topic 27 → ip, ss, vẫn, hơn, thua
Topic 7 → sạc, pin, nhanh, không, dùng
Topic 31 → nhôm, titan, khung, thép, viền
Topic 6 → may, co, ban, quá, đầu
Topic 11 → x200, vivo, mini, pro, x200pro
Topic 28 → positive, quá, đẹp, ơn, anh
Topic 26 → xiaomi, huawei, đỉnh, đẹp, hơn
Topic 35 → gen, 685, snap, chip, mạnh
Topic 10 → oppo, vivo, hơn, chụp, nó
Topic 17 → ram, bản, 256, 256gb, 128
Topic 8 → fold, flip, gập, n3, find
Topic 22 → dán, ốp, màn, cong, cường
Topic 29 → màu, đẹp, trắng, xanh, 

In [14]:
# Lọc top 15 topic có số lượng cao nhất
top_15_topics = topic_info.head(15)

# In ra các mô tả cho top 15 topic
for topic_id in top_15_topics["Topic"]:
    # Lấy các từ trong topic
    words = topic_model.get_topic(topic_id)
    
    # Kiểm tra xem topic có từ khóa không rõ ràng
    if not words:
        description = "Không có mô tả"
    else:
        # Lấy 5 từ đầu tiên làm mô tả cho topic
        description = ", ".join([word[0] for word in words[:5]]) 
        
    print(f"Topic {topic_id} → {description}")

Topic -1 → không, có, neutral, mà, mua
Topic 0 → pro, 15, 16, 14, 13
Topic 1 → s23, ultra, s24, s25, s23u
Topic 2 → bác, không, đâu, mà, gì
Topic 3 → iphone, android, apple, ip, samsung
Topic 4 → neutral, tiền, không, có, mà
Topic 5 → video, camera, chụp, review, ảnh
Topic 6 → may, co, ban, quá, đầu
Topic 7 → sạc, pin, nhanh, không, dùng
Topic 8 → fold, flip, gập, n3, find
Topic 9 → x8, find, oppo, pro, x7
Topic 10 → oppo, vivo, hơn, chụp, nó
Topic 11 → x200, vivo, mini, pro, x200pro
Topic 12 → xiaomi, phần, không, nó, hãng
Topic 13 → samsung, sam, sony, sung, samfan
